# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. My lane and why

**Provisional lane: Refresh / Content Opportunity Scoring**

I want to investigate which content pages should be reviewed first for a potential refresh, expansion, protection, pruning, or monitoring action. I chose this lane because the starter dataset contains page-level signals about content age, recent updates, search impressions, clicks, sessions, and search position. These signals give a useful starting point for identifying observable differences between pages and building a ranked review queue. My goal is not to automatically decide what should be changed, but to investigate whether data and ML can help prioritize the pages that deserve human attention first.

In [5]:
# Check the unit of analysis and available signals for the chosen lane.

import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique():,}")

print("\nSelected signals:")
print(
    df[
        [
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "clicks_90d",
            "avg_position",
            "trend_direction",
        ]
    ].head()
)

Rows: 30,000
Columns: 44
Unique content items: 30,000
Unique clients: 32

Selected signals:
   content_age_days  days_since_last_update  impressions_90d  clicks_90d  \
0               187                      20             3803          29   
1               445                      25            15320           7   
2               141                      20            12581          11   
3               463                      22            11751          58   
4               263                      14            19140          24   

   avg_position trend_direction  
0          10.6            down  
1          20.3            down  
2          36.5            down  
3           6.2          stable  
4          44.0            down  


## 2. The question: decision, action, cost of a wrong call

### Research question

**Which pages should be reviewed first for a potential content action, based on observable search performance, content age, freshness, and related signals?**

### Decision

The decision is **which pages to prioritize for human review** when review capacity is limited.

### Action

A reviewer could investigate a high-priority page for an appropriate action such as refreshing, expanding, protecting, pruning, or monitoring it. The eventual output would support prioritization rather than automatically deciding the action.

### Cost of a wrong call

A false positive could spend limited review time on a page that does not represent a meaningful opportunity. A false negative could cause a potentially valuable page to be overlooked while other pages are reviewed.

The aim is therefore to improve the ordering of pages for human review, rather than guarantee that a particular intervention will improve performance.

In [6]:
# Check the main decision-support signals available at page level.

signal_summary = pd.DataFrame({
    "signal": [
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "avg_position"
    ],
    "non_null_rows": [
        df["content_age_days"].notna().sum(),
        df["days_since_last_update"].notna().sum(),
        df["impressions_90d"].notna().sum(),
        df["clicks_90d"].notna().sum(),
        df["sessions_90d"].notna().sum(),
        df["avg_position"].notna().sum()
    ]
})

display(signal_summary)

,signal,non_null_rows
0,content_age_days,30000
1,days_since_last_update,30000
2,impressions_90d,30000
3,clicks_90d,30000
4,sessions_90d,30000
5,avg_position,30000


## 3. Quick look at the data (2–3 real numbers)

The starter exploration gives several early signals that make this lane worth investigating.

* Pages labeled **down** had a median content age of **216 days**, while pages labeled **up** had a median age of **291.5 days**. The difference was **75.5 days**.
* In my held-out experiment, the readable decision tree achieved **0.750 Precision@20**, compared with **0.500** for the hand-built rule.
* At Precision@50, the decision tree achieved **0.680**, compared with **0.540** for the hand-built rule.

These are initial observations from the starter dataset and a single held-out experiment. They suggest that content age and other page-level signals may contain useful information for prioritization, but they do not establish causation or guarantee that the same performance will hold under stronger validation.

In [7]:
# Reproduce the key numbers from the starter dataset.

age_summary = (
    df.groupby("trend_direction")["content_age_days"]
      .agg(["count", "median", "mean"])
      .round(1)
)

print("Content age by trend direction:")
display(age_summary)

median_age = df.groupby("trend_direction")["content_age_days"].median()

age_difference = median_age["down"] - median_age["up"]

print(
    f"Median age difference (down - up): "
    f"{age_difference:.1f} days"
)

print("\nStarter dataset size:")
print(f"{len(df):,} content items")

Content age by trend direction:


,count,median,mean
trend_direction,,,
down,16262,216.0,236.2
flat,1152,231.0,245.9
new,2236,279.0,238.7
stable,5962,300.0,295.4
up,4388,291.5,288.5


Median age difference (down - up): -75.5 days

Starter dataset size:
30,000 content items


## 4. Careful words: what I can and can't claim

### What I can claim

I can report observed relationships and measured results from the starter dataset. For example, I observed that the `up` and `down` groups had different median content ages, and that the simple decision tree performed better than the hand-built rule on my held-out test split.

As the project develops, I can test whether these signals remain useful under stronger validation and whether they can support a ranked page-review queue.

### What I cannot claim

I cannot claim that older content causes better performance, that refreshing a particular page will cause recovery, or that the model predicts Google's ranking algorithm.

The starter `trend_direction` label describes a recent change in impressions; it is not a future outcome. For later modeling, I will need to define the feature and target windows carefully so that information from the outcome period does not leak into the features.

The intended output is decision support: evidence that helps a human prioritize pages for review, not an automatic decision about what will succeed.

In [8]:
# Sanity checks for the framing:
# trend_direction is the observed label source,
# while content_id/client_id are identifiers rather than features.

print("Unique clients:", df["client_id"].nunique())
print("Unique content items:", df["content_id"].nunique())

print("\nObserved trend distribution:")
display(
    df["trend_direction"]
      .value_counts()
      .rename_axis("trend_direction")
      .to_frame("count")
)

print("\nColumns that must not be used as model features:")
print(["content_id", "client_id", "trend_direction", "trend_pct"])

Unique clients: 32
Unique content items: 30000

Observed trend distribution:


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Columns that must not be used as model features:
['content_id', 'client_id', 'trend_direction', 'trend_pct']


## Self-check

Before submitting:

* [x] Every section above is filled with both reasoning and supporting code.
* [x] The notebook runs from top to bottom without errors.
* [x] The unit of analysis is clearly defined as a content page.
* [x] The decision, possible action, and cost of a wrong recommendation are clear.
* [x] The lane is one of the four predefined lanes.
* [x] At least two real numbers from the starter dataset support the lane choice.
* [x] Claims use careful language such as observed, measured, directional, and decision-support.
* [x] I do not claim causation or that the model predicts Google's ranking algorithm.
* [x] I understand that `trend_direction` is an observed label source and should not be used as a model feature.
* [x] No client names, private URLs, private queries, or other sensitive information are included.
* [x] The completed notebook will be committed under `work/notebooks/w01_research_question.ipynb`.